# Bayesian Optimization

Bayesian Optimization is a global optimization method for finding the optimal value of an expensive-to-evaluate function.

This method works by:
* Building a probabilistic model (surrogate model) for the objective function, usually using a Gaussian Process
* Using an acquisition function to determine the next evaluation point
* Updating the model based on the new evaluation results
* Repeating the process until convergence or reaching an iteration limit
* The main advantage of Bayesian Optimization is sample efficiency, meaning it can find optimal solutions with a relatively small number of experiments compared to grid search or random search
* In the context of machine learning, this method is commonly used for model hyperparameter optimization

## Import Library

This section aims to import all the modules required for the optimization process. The libraries used include:

* Bayesian optimization module (BayesSearchCV)
* Parameter space definition (Real, Integer)
* Dataset and data sharing utilities
* Classification model (XGBoost)
* Evaluation method (accuracy)
* This stage is technical preparation before the modeling process begins

In [ ]:
!pip install scikit-optimize

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 4.1 MB/s eta 0:00:00


In [ ]:
import numpy as np
from skopt import BayesSearchCV
from skopt.space import Real, Integer
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier

## Creating Dataset

The dataset is loaded and separated into:

* Independent variable (feature) → X
* Dependent variable (target) → y

The purpose of this section is to provide the data that will be used to train and evaluate the classification model.

In [ ]:
data = load_breast_cancer()
X = data.data
y = data.target

## Train-Test Split

The data is divided into two subsets:

* Training data (training set)
* Testing data (testing set)

The goal is to ensure that model performance evaluation is performed on data not used during the training process, thereby reducing the risk of overfitting and increasing the validity of the results.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

## Define Search Space

The search space defines the range of values ​​that the Bayesian Optimization algorithm can explore for each hyperparameter.

The optimized parameters include:
* max_depth
* learning_rate
* n_estimators

The purpose of this stage is to limit the search space to ensure the optimization process remains focused and efficient.

| Parameter     | Makna             | Range    |
| ------------- | ----------------- | -------- |
| max_depth     | Kedalaman tree    | 3–10     |
| learning_rate | Kecepatan belajar | 0.01–0.3 |
| n_estimators  | Jumlah tree       | 50–300   |


In [ ]:
search_space = {
    'max_depth': Integer(3, 10),
    'learning_rate': Real(0.01, 0.3, prior='log-uniform'),
    'n_estimators': Integer(50, 300)
}

## Model Initiation

The XGBoost model is initialized without optimal hyperparameter configuration.

The goal of this step is to provide a baseline estimator for optimization.

In [ ]:
model = XGBClassifier(
    use_label_encoder=False,
    eval_metric='logloss'
)

## Bayesian Optimization

BayesSearchCV is configured with:

* Estimator (model)
* Search space
* Number of iterations (n_iter)
* Cross-validation (cv)
* Parallelization settings

The purpose of this section is to configure the optimization mechanism, including the number of experiments and the evaluation method used in each iteration.

In [ ]:
opt = BayesSearchCV(
    estimator=model,
    search_spaces=search_space,
    n_iter=30,
    cv=3,
    random_state=42,
    n_jobs=-1
)

🎯 Objective:

Tell the Bayesian Optimization engine to:
* estimator → model to be tuned
* search_spaces → parameter space
* n_iter=30 → try 30 combinations
* cv=3 → 3-fold cross validation
* n_jobs=-1 → use all CPU cores

👉 This is the core of Bayesian Optimization.

Here's what happens:
* Create a surrogate model (Gaussian Process)
* Choose the first parameter
* Evaluate
* Update the probabilistic model
* Choose the next parameter
* Repeat for 30 iterations

## Optimization Process

The fit() method is run to start the Bayesian Optimization process.

In this stage:

The model is trained with the initial parameter combination.
Performance is evaluated.
The probabilistic model is updated.
The next parameter combination is selected based on the acquisition function.
This process continues until the number of iterations is reached.

In [ ]:
opt.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [03:01:16] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


BayesSearchCV(cv=3,
              estimator=XGBClassifier(base_score=None, booster=None,
                                      callbacks=None, colsample_bylevel=None,
                                      colsample_bynode=None,
                                      colsample_bytree=None, device=None,
                                      early_stopping_rounds=None,
                                      enable_categorical=False,
                                      eval_metric='logloss', feature_types=None,
                                      feature_weights=None, gamma=None,
                                      grow_policy=None, importance_type=None,
                                      interaction_constrain...
                                      multi_strategy=None, n_estimators=None,
                                      n_jobs=None, num_parallel_tree=None, ...),
              n_iter=30, n_jobs=-1, random_state=42,
              search_spaces={'learning_rate': Real(low=0.01, high=0.3, prior='log-uniform', transform='normalize'),
                             'max_depth': Integer(low=3, high=10, prior='uniform', transform='normalize'),
                             'n_estimators': Integer(low=50, high=300, prior='uniform', transform='normalize')})

## Best Parameter Identification

After the optimization process is complete, the best_params_ attribute is used to obtain the best-performing hyperparameter combination based on cross-validation

The goal is to determine the optimal model configuration within a predefined search space

In [ ]:
print("Best Parameters:", opt.best_params_)

Best Parameters: OrderedDict({'learning_rate': 0.1214083024909187, 'max_depth': 10, 'n_estimators': 91})


## Model Evaluation

Model terbaik digunakan untuk melakukan prediksi pada data pengujian. Selanjutnya dihitung nilai akurasi sebagai ukuran performa akhir

Tahap ini bertujuan untuk mengukur generalisasi model terhadap data baru yang belum pernah dilihat sebelumnya

In [ ]:
y_pred = opt.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print("Test Accuracy:", accuracy)

Test Accuracy: 0.956140350877193


# Bayesian Optimization with Optuna

In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 3.6 MB/s eta 0:00:00


# Install and Import Library

In [ ]:
import optuna
import sklearn.datasets
import sklearn.model_selection
import sklearn.ensemble
import sklearn.metrics

# Load Dataset

In [ ]:
# Load breast cancer dataset
data = sklearn.datasets.load_breast_cancer()
X = data.data
y = data.target

# Split data
X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Objective Function

In [ ]:
def objective(trial):

    # Hyperparameter search space
    n_estimators = trial.suggest_int("n_estimators", 50, 300)
    max_depth = trial.suggest_int("max_depth", 2, 32)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 10)

    # Model
    model = sklearn.ensemble.RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        random_state=42
    )

    # Cross-validation
    score = sklearn.model_selection.cross_val_score(
        model, X_train, y_train, cv=3, scoring="accuracy"
    )

    return score.mean()

# Running Bayesian Optimization (Optuna)

In [ ]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30)

[I 2026-02-26 04:49:11,811] A new study created in memory with name: no-name-2504a7b3-b577-4573-82de-a2f7ecb41bfa
[I 2026-02-26 04:49:12,480] Trial 0 finished with value: 0.9538166608574415 and parameters: {'n_estimators': 99, 'max_depth': 20, 'min_samples_split': 9}. Best is trial 0 with value: 0.9538166608574415.
[I 2026-02-26 04:49:13,592] Trial 1 finished with value: 0.9516236784013014 and parameters: {'n_estimators': 171, 'max_depth': 8, 'min_samples_split': 9}. Best is trial 0 with value: 0.9538166608574415.
[I 2026-02-26 04:49:15,248] Trial 2 finished with value: 0.9516236784013014 and parameters: {'n_estimators': 160, 'max_depth': 24, 'min_samples_split': 10}. Best is trial 0 with value: 0.9538166608574415.
[I 2026-02-26 04:49:17,957] Trial 3 finished with value: 0.9472377134890205 and parameters: {'n_estimators': 175, 'max_depth': 9, 'min_samples_split': 10}. Best is trial 0 with value: 0.9538166608574415.
[I 2026-02-26 04:49:20,544] Trial 4 finished with value: 0.947237713489

# Best Hyperparameter

In [ ]:
print("Best trial:")
print("  Accuracy:", study.best_trial.value)
print("  Params: ")
for key, value in study.best_trial.params.items():
    print(f"    {key}: {value}")

Best trial:
  Accuracy: 0.956009643313582
  Params: 
    n_estimators: 106
    max_depth: 19
    min_samples_split: 5


# Evaluation Model

In [ ]:
best_params = study.best_trial.params

best_model = sklearn.ensemble.RandomForestClassifier(
    **best_params,
    random_state=42
)

best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)

accuracy = sklearn.metrics.accuracy_score(y_test, y_pred)
print("Test Accuracy:", accuracy)

Test Accuracy: 0.9649122807017544
